# ADiCo-IRL — System Test Notebook

Run each block independently to verify the system before a full deployment.

| Block | Tests |
|---|---|
| 1 | Vector robot connectivity + battery |
| 2 | Camera capture + warped top-down view |
| 3 | Movement controller replay from `robot_trajectory.csv` |

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../adico_irl')) 

import math, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # safe for Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

print('Imports OK')

Imports OK


---
## Block 1 — Robot Connectivity
Connects to each Vector, reads battery level, and makes the robot say hello.
Edit `SERIALS` to match your robots.

In [2]:
import anki_vector

SERIALS = [
    "00a132eb",   # Robot 0
    "00506502",   # Robot 1
    "00a12dbd",   # Robot 2
]

SAY_HELLO = True   # set False to skip speech (faster)

results = []
for idx, serial in enumerate(SERIALS):
    try:
        robot = anki_vector.Robot(serial, behavior_activation_timeout=30)
        robot.connect()
        battery = robot.get_battery_state()
        heading = robot.pose.rotation.angle_z.degrees
        pos = robot.pose.position
        print(f"✓ Robot {idx}  serial={serial}")
        print(f"   battery level : {battery.battery_level}")
        print(f"   battery volts : {battery.battery_volts:.2f} V")
        print(f"   pose          : x={pos.x:.1f} y={pos.y:.1f} heading={heading:.1f}°")
        if SAY_HELLO:
            robot.behavior.say_text(f"Hello, I am robot {idx}")
        robot.motors.set_wheel_motors(0, 0)   # ensure stopped
        robot.disconnect()
        results.append({'serial': serial, 'ok': True, 'battery': battery.battery_level})
    except Exception as e:
        print(f"✗ Robot {idx}  serial={serial}  ERROR: {e}")
        results.append({'serial': serial, 'ok': False, 'battery': None})

ok_count = sum(r['ok'] for r in results)
print(f"\n{'='*40}")
print(f"Connected: {ok_count}/{len(SERIALS)} robots")

22:15:43.971     connection.Connection    INFO  Connecting to 10.0.0.30:443 for Vector-G7P7 using /home/svarp/.anki_vector/Vector-G7P7-00a132eb.cert
22:15:44.508     connection.Connection    INFO  BehaviorControl control_granted_response { }


✓ Robot 0  serial=00a132eb
   battery level : 3
   battery volts : 4.07 V
   pose          : x=-1.7 y=-0.2 heading=-9.0°


22:15:47.613     connection.Connection    INFO  Connecting to 10.0.0.148:443 for Vector-S9G7 using /home/svarp/.anki_vector/Vector-S9G7-00506502.cert
22:15:48.064     connection.Connection    INFO  BehaviorControl control_granted_response { }


✓ Robot 1  serial=00506502
   battery level : 3
   battery volts : 4.11 V
   pose          : x=0.0 y=0.0 heading=-2.4°


22:15:51.845     connection.Connection    INFO  Connecting to 10.0.0.144:443 for Vector-X9U9 using /home/svarp/.anki_vector/Vector-X9U9-00a12dbd.cert
22:15:52.533     connection.Connection    INFO  BehaviorControl control_granted_response { }


✓ Robot 2  serial=00a12dbd
   battery level : 2
   battery volts : 4.09 V
   pose          : x=-4.0 y=0.4 heading=-3.8°

Connected: 3/3 robots


---
## Block 2 — Camera Check
Captures a frame, shows the raw image and the warped top-down view.
Requires `calibrate.py` to have been run (so `arena_corners.npy` exists).

In [48]:
import yaml
from pathlib import Path
from perception.camera import Camera
from utils.perspective import load_corners, compute_homography, warp_frame

ROOT = Path('..').resolve()

with open(ROOT / 'adico_irl' / 'config.yaml') as f:
    config = yaml.safe_load(f)

# ── capture frame ──
cam = Camera(config)
frame = cam.get_frame()
cam.release()
print(f"Frame shape: {frame.shape}  dtype: {frame.dtype}")

# ── warp if corners exist ──
# corners = load_corners(ROOT / config['corners_path')

has_corners = corners is not None
if has_corners:
    H = compute_homography(corners, config['warped_px'])
    warped = warp_frame(frame, H, config['warped_px'])
    print(f"Warped shape : {warped.shape}")
else:
    print("WARNING: arena_corners.npy not found. Run calibrate.py first.")
    warped = None

# ── plot ──
ncols = 2 if has_corners else 1
fig, axes = plt.subplots(1, ncols, figsize=(14, 5))
if ncols == 1:
    axes = [axes]

axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Raw Camera Frame')
axes[0].axis('off')

if has_corners:
    # draw corners on raw frame
    corner_labels = ['TL','TR','BR','BL']
    ax_raw = axes[0]
    for i, (cx, cy) in enumerate(corners):
        ax_raw.plot(cx, cy, 'ro', ms=8)
        ax_raw.text(cx+10, cy-10, corner_labels[i], color='red', fontsize=9, fontweight='bold')

    axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Warped Top-Down View')
    axes[1].axis('off')

plt.tight_layout()
plt.savefig('test_camera.png', dpi=120)
plt.show()
print('Saved → test_camera.png')

Frame shape: (720, 1280, 3)  dtype: uint8
Warped shape : (600, 600, 3)
Saved → test_camera.png


/tmp/ipykernel_7412/3221341727.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Block 3 — Movement Controller: Fixed Trajectory Replay

Loads `checkpoints/robot_trajectory.csv` (`timestamp, linear_x, angular_z`)  
and converts it to world-frame `(vx, vy)` via dead-reckoning heading integration.

**Simulation mode** (`ROBOT_SERIAL = None`): plots the commanded trajectory only.  
**Real robot mode** (`ROBOT_SERIAL = "00506502"`): also sends wheel commands and overlays odometry.

In [4]:
# ── Config ────────────────────────────────────────────────────────────────────
TRAJ_CSV     = 'checkpoints/robot_trajectory.csv'
ROBOT_SERIAL = None      # set to e.g. "00506502" to run on real robot
SPEED_SCALE  = 1.0       # scale linear speed (reduce if robot moves too fast)

# Controller params (must match config.yaml)
WHEEL_BASE_MM      = 45.0
MAX_WHEEL_SPEED_MM = 150.0
TURN_WHEEL_MM      = 100.0
MIN_TURN_RAD       = 0.087
U_RANGE            = 0.5

# ── Load trajectory ───────────────────────────────────────────────────────────
df = pd.read_csv(TRAJ_CSV)
print(f"Loaded {len(df)} steps from {TRAJ_CSV}")
print(df.head())

Loaded 1972 steps from checkpoints/robot_trajectory.csv
      timestamp  linear_x  angular_z
0  1.778637e+09       0.0        0.0
1  1.778637e+09       0.0        0.0
2  1.778637e+09       0.0        0.0
3  1.778637e+09       0.0        0.0
4  1.778637e+09       0.0        0.0


In [5]:
# ── Convert (linear_x, angular_z) → world-frame (vx, vy) ──────────────────────
# Integrate angular_z to get robot heading, then rotate linear_x into world frame.

timestamps = df['timestamp'].values
linear_x   = df['linear_x'].values * SPEED_SCALE
angular_z  = df['angular_z'].values

heading = 0.0
vx_list, vy_list = [], []

for i in range(len(timestamps)):
    if i > 0:
        dt = timestamps[i] - timestamps[i - 1]
        heading += angular_z[i] * dt
        heading = (heading + math.pi) % (2 * math.pi) - math.pi   # wrap to [-π, π]
    vx_list.append(linear_x[i] * math.cos(heading))
    vy_list.append(linear_x[i] * math.sin(heading))

vx_arr = np.array(vx_list)
vy_arr = np.array(vy_list)

# Dead-reckoning XY trajectory from the CSV
cmd_x, cmd_y = [0.0], [0.0]
for i in range(1, len(timestamps)):
    dt = timestamps[i] - timestamps[i - 1]
    cmd_x.append(cmd_x[-1] + vx_arr[i] * dt)
    cmd_y.append(cmd_y[-1] + vy_arr[i] * dt)

print(f"Trajectory span: x=[{min(cmd_x):.3f}, {max(cmd_x):.3f}] m  "
      f"y=[{min(cmd_y):.3f}, {max(cmd_y):.3f}] m")

# Quick preview plot of the CSV trajectory
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(cmd_x, cmd_y, 'b-', lw=1.5)
axes[0].plot(cmd_x[0], cmd_y[0], 'go', ms=8, label='start')
axes[0].plot(cmd_x[-1], cmd_y[-1], 'rs', ms=8, label='end')
axes[0].set_title('Dead-Reckoning Trajectory'); axes[0].legend()
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_aspect('equal'); axes[0].grid(True)

t_rel = timestamps - timestamps[0]
axes[1].plot(t_rel, linear_x, 'b-'); axes[1].set_title('linear_x (m/s)')
axes[1].set_xlabel('time (s)'); axes[1].grid(True)
axes[2].plot(t_rel, angular_z, 'r-'); axes[2].set_title('angular_z (rad/s)')
axes[2].set_xlabel('time (s)'); axes[2].grid(True)

plt.tight_layout()
plt.savefig('test_trajectory_preview.png', dpi=120)
plt.show()
print('Saved → test_trajectory_preview.png')

Trajectory span: x=[-0.279, 0.636] m  y=[-0.760, 0.155] m
Saved → test_trajectory_preview.png


/tmp/ipykernel_35183/2371768957.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# ── Run on robot (or simulate) ─────────────────────────────────────────────────
# Real robot: set ROBOT_SERIAL above. Simulation: leave it as None.

def _angle_diff(target, current):
    d = (target - current) % (2 * math.pi)
    if d > math.pi:
        d -= 2 * math.pi
    return d

def _send_action(robot, vx, vy, dt):
    """Turn-then-drive for one timestep. robot=None → simulation."""
    v = math.hypot(vx, vy)
    if v < 1e-4:
        if robot:
            robot.motors.set_wheel_motors(0, 0)
        return

    theta_target = math.atan2(vy, vx)

    if robot:
        current_heading = robot.pose.rotation.angle_z.radians
        delta = _angle_diff(theta_target, current_heading)
        if abs(delta) > MIN_TURN_RAD:
            omega = 2 * TURN_WHEEL_MM / WHEEL_BASE_MM
            turn_t = abs(delta) / omega
            lw = -TURN_WHEEL_MM if delta > 0 else TURN_WHEEL_MM
            robot.motors.set_wheel_motors(lw, -lw)
            time.sleep(turn_t)
            robot.motors.set_wheel_motors(0, 0)
            time.sleep(0.05)
        spd = v * MAX_WHEEL_SPEED_MM / U_RANGE
        robot.motors.set_wheel_motors(spd, spd)
        time.sleep(dt)
        robot.motors.set_wheel_motors(0, 0)

# ── connect ──
robot = None
if ROBOT_SERIAL:
    try:
        import anki_vector
        robot = anki_vector.Robot(ROBOT_SERIAL, behavior_activation_timeout=30)
        robot.connect()
        robot.behavior.set_head_angle(anki_vector.util.degrees(10))
        robot.behavior.set_lift_height(0.0)
        print(f'Connected to {ROBOT_SERIAL}')
    except Exception as e:
        print(f'Connection failed: {e} — running in simulation mode')
        robot = None
else:
    print('Simulation mode (no robot serial set)')

# ── replay loop ──
odom_x, odom_y = [], []
step_times = []

try:
    for i in range(len(timestamps)):
        vx = vx_arr[i]
        vy = vy_arr[i]
        dt = (timestamps[i] - timestamps[i - 1]) if i > 0 else 0.1

        t0 = time.time()
        _send_action(robot, vx, vy, dt)

        if robot:
            odom_x.append(robot.pose.position.x / 1000.0)
            odom_y.append(robot.pose.position.y / 1000.0)

        step_times.append(time.time() - t0)

        if i % 20 == 0:
            print(f'  step {i:4d}/{len(timestamps)}  vx={vx:+.3f}  vy={vy:+.3f}  '
                  f'elapsed={step_times[-1]*1000:.0f} ms')

except KeyboardInterrupt:
    print('Interrupted by user')
finally:
    if robot:
        robot.motors.set_wheel_motors(0, 0)
        robot.disconnect()
        print('Robot disconnected')

print(f'\nDone. {len(step_times)} steps, avg step time: {np.mean(step_times)*1000:.1f} ms')

Simulation mode (no robot serial set)
  step    0/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step   20/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step   40/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step   60/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step   80/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  100/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  120/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  140/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  160/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  180/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  200/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  220/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  240/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  260/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  280/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  300/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  320/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  340/1972  vx=+0.000  vy=+0.000  elapsed=0 ms
  step  

In [7]:
# ── Final comparison plot ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Movement Controller — Trajectory Replay', fontsize=13)

# XY trajectory
ax = axes[0]
ax.plot(cmd_x, cmd_y, 'b-', lw=1.5, label='Commanded (dead-reckoning)')
if odom_x:
    ax.plot(odom_x, odom_y, 'r-', lw=1.5, label='Actual odometry')
ax.plot(cmd_x[0], cmd_y[0], 'go', ms=10, zorder=5, label='Start')
ax.plot(cmd_x[-1], cmd_y[-1], 'rs', ms=10, zorder=5, label='End')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('XY Trajectory'); ax.legend(); ax.grid(True)
ax.set_aspect('equal')

# Speed over time
ax2 = axes[1]
speed = np.hypot(vx_arr, vy_arr)
t_rel = timestamps - timestamps[0]
ax2.plot(t_rel, speed, 'b-', label='Speed |v| (m/s)')
ax2.plot(t_rel, vx_arr, 'g--', alpha=0.7, label='vx')
ax2.plot(t_rel, vy_arr, 'r--', alpha=0.7, label='vy')
ax2.set_xlabel('time (s)'); ax2.set_ylabel('velocity (m/s)')
ax2.set_title('Velocity Profile'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('test_movement_controller.png', dpi=120)
plt.show()
print('Saved → test_movement_controller.png')

Saved → test_movement_controller.png


/tmp/ipykernel_35183/318608590.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
